In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 读取数据
train_df = pd.read_csv('unsw-nb15/UNSW_NB15_training-set.csv')
test_df = pd.read_csv('unsw-nb15/UNSW_NB15_testing-set.csv')

# 移除无关列
train_df.drop(columns=['id', 'label'], inplace=True)
test_df.drop(columns=['id', 'label'], inplace=True)

# 检查是否有缺失值
print("Train missing values:", train_df.isnull().sum().sum())
print("Test missing values:", test_df.isnull().sum().sum())

# 需要进行 One-hot 编码的列
cols = ['proto', 'state', 'service']

# One-hot 编码函数
def one_hot_encode(df, cols):
    return pd.get_dummies(df, columns=cols, drop_first=False)

# 合并数据进行 One-hot 编码
combined_data = pd.concat([train_df, test_df])
combined_data = one_hot_encode(combined_data, cols)

# 分离特征和目标变量
target = combined_data.pop('attack_cat')

# 归一化函数
def normalize(df):
    scaler = StandardScaler()
    df[:] = scaler.fit_transform(df)
    return df

# 归一化数据
normalized_data = normalize(combined_data)

# 添加目标列
normalized_data["Class"] = target

# 分离特征和标签
X = normalized_data.drop(columns=['Class']).values
y = normalized_data['Class'].values
print(X.shape)
# 进行 One-hot 编码
encoder = OneHotEncoder(sparse=False)
y_onehot = encoder.fit_transform(y.reshape(-1, 1))

# 转换为 PyTorch 张量
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y_onehot, dtype=torch.float32)


Train missing values: 0
Test missing values: 0
(257673, 196)


E:\shendu\anaconda3\envs\pytorch1\lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [2]:
import torch
import torch.nn as nn

class ASTP(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        # 多尺度时间卷积（避免小卷积核）
        self.temp_conv = nn.ModuleList([
            nn.Conv1d(in_channels, 16, 32, padding='same'),  # 空洞率2
            nn.Conv1d(in_channels, 16, 64, padding='same', dilation=2),
            nn.Conv1d(in_channels, 16, 96, padding='same', dilation=4)
        ])
        # 特征动态融合（参考IEEE TDSC 2023）
        self.fusion_gate = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(48, 48, 1),
            nn.Sigmoid()
        )
        # 空间注意力（参考ACM CCS 2022）
        self.spatial_att = nn.Sequential(
            nn.Conv1d(48, 48, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        branch_outs = [conv(x) for conv in self.temp_conv]
        merged = torch.cat(branch_outs, dim=1)  # [B,48,L]
        # 动态特征融合
        fused = merged * self.fusion_gate(merged)  # [B,48,L]
        # 空间注意力增强
        return fused * self.spatial_att(fused)

class TRGU(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        # 门控机制（参考ICML 2023）
        self.gate = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.Sigmoid()
        )
        # 残差时序卷积（替代LSTM）
        self.res_conv = nn.Sequential(
            nn.Conv1d(feat_dim, feat_dim, 64, padding='same'),
            nn.BatchNorm1d(feat_dim),
            nn.GELU()
        )
        # 时间注意力
        self.time_att = nn.MultiheadAttention(feat_dim, 4, batch_first=True)

    def forward(self, x):
        # x shape: [B,L,D]
        # 残差路径
        residual = x
        # 时序卷积
        x_conv = self.res_conv(x.permute(0,2,1)).permute(0,2,1)  # [B,L,D]
        # 时间注意力
        x_att, _ = self.time_att(x, x, x)  # [B,L,D]
        # 门控融合
        gate = self.gate(x)
        return residual + gate*x_conv + (1-gate)*x_att

class NSLKDDModel(nn.Module):
    def __init__(self, input_dim=196, num_classes=10):
        super().__init__()
        
        # 特征提取层
        self.astp = ASTP(in_channels=1)
        self.pool = nn.AdaptiveMaxPool1d(input_dim//4)
        self.bn = nn.BatchNorm1d(48)
        
        # 时序建模
        self.trgu = nn.Sequential(
            TRGU(48),
            TRGU(48),
            nn.TransformerEncoder(
                encoder_layer=nn.TransformerEncoderLayer(
                    d_model=48,
                    nhead=4,
                    dim_feedforward=192,
                    batch_first=True
                ),
                num_layers=2
            )
        )
        
        # 分类头（改进自IEEE TDSC 2023）
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(48, 128),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # 输入形状: [B,1,122]
        x = self.astp(x)          # [B,48,122]
        x = self.pool(x)          # [B,48,30]
        x = self.bn(x)            # 特征标准化
        x = x.permute(0,2,1)      # [B,30,48]
        x = self.trgu(x)          # [B,30,48]
        x = x.permute(0,2,1)      # [B,48,30]
        return self.classifier(x) # [B,num_classes]


In [3]:
from sklearn import metrics
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
# 设置超参数
batch_size = 64
epochs = 20    #15轮 84.16左右
learning_rate = 0.0008
k=10
# 交叉验证
kfold = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
oversample = RandomOverSampler(sampling_strategy='minority')

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)  # 预测的概率
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

criterion = FocalLoss()
# 统计结果
oos_pred = []
# 初始化结果列表
oos_accuracies = []
last_fold_y_true = []
last_fold_y_pred = []

# 初始化模型
model = NSLKDDModel(input_dim=196, num_classes=10).to(device)
#criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 遍历折叠
for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y), start=1):

    # 划分数据集
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # 过采样少数类
    X_train, y_train = oversample.fit_resample(X_train, y_train)

    # 使用 LabelEncoder 转换 y_train 和 y_test
    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(y_train)  # 转换成整数标签
    y_test = label_encoder.transform(y_test)

    # 确保 y_train 和 y_test 是 numpy 数组的 int 类型
    y_train = y_train.astype(np.int64)
    y_test = y_test.astype(np.int64)

    # 转换为 PyTorch 数据集
    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

    # 加载数据
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0

        # 使用 tqdm 显示进度条
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch_data, batch_labels in progress_bar:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)

            batch_data = batch_data.unsqueeze(1)  # 添加通道维度
            optimizer.zero_grad()
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            # 累积损失和准确性
            epoch_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == batch_labels).sum().item()
            total += batch_labels.size(0)

            # 更新进度条
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        # 计算每轮的平均损失和准确率
        epoch_loss /= len(train_loader)
        epoch_acc = correct / total
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")    

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch_data, batch_labels in test_loader:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)
            batch_data = batch_data.unsqueeze(1)  # 添加通道维度
            outputs = model(batch_data)
            _, preds = torch.max(outputs, 1)

            y_true.extend(batch_labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # 测试集各类别数量
    # test_class_counts = Counter(y_true)
    # print("\nTest Set Class Distribution:")
    # for label, count in sorted(test_class_counts.items()):
    #     print(f"  Class {label}: {count}")

    # 计算准确率
    acc = metrics.accuracy_score(y_true, y_pred)
    oos_accuracies.append(acc)
    print(f"Fold {fold} Accuracy: {acc:.4f}")

    # 保存最后一折的结果
    if fold == kfold.get_n_splits():
        last_fold_y_true = y_true
        last_fold_y_pred = y_pred

# 保存模型
model_save_path = "modelU8.pth"
torch.save(model, model_save_path)
print(f"Complete model saved to {model_save_path}")

# 输出每一折的准确率
print("Fold Accuracies:")
for i, acc in enumerate(oos_accuracies, start=1):
    print(f"  Fold {i}: {acc:.4f}")
 

Epoch 1/20:   0%|          | 0/4929 [00:00<?, ?it/s]E:\shendu\anaconda3\envs\pytorch1\lib\site-packages\torch\nn\modules\conv.py:306: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\Convolution.cpp:1041.)
  return F.conv1d(input, weight, bias, self.stride,
E:\shendu\anaconda3\envs\pytorch1\lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
Epoch 1/20: 100%|██████████| 4929/4929 [00:52<00:00, 93.44it/s, loss=0.3568] 


Epoch 1/20 - Loss: 0.3117, Accuracy: 0.8041


Epoch 2/20: 100%|██████████| 4929/4929 [00:47<00:00, 103.56it/s, loss=0.3238]


Epoch 2/20 - Loss: 0.2420, Accuracy: 0.8309


Epoch 3/20: 100%|██████████| 4929/4929 [00:47<00:00, 104.12it/s, loss=0.1463]


...

Epoch 18/20 - Loss: 0.1310, Accuracy: 0.8853


Epoch 19/20: 100%|██████████| 4929/4929 [00:49<00:00, 98.97it/s, loss=0.1436] 


Epoch 19/20 - Loss: 0.1313, Accuracy: 0.8852


Epoch 20/20: 100%|██████████| 4929/4929 [00:50<00:00, 98.50it/s, loss=0.1645] 


Epoch 20/20 - Loss: 0.1344, Accuracy: 0.8836
Fold 10 Accuracy: 0.8328
Complete model saved to modelU8.pth
Fold Accuracies:
  Fold 1: 0.8125
  Fold 2: 0.8131
  Fold 3: 0.8097
  Fold 4: 0.8172
  Fold 5: 0.8233
  Fold 6: 0.8247
  Fold 7: 0.8253
  Fold 8: 0.8278
  Fold 9: 0.8335
  Fold 10: 0.8328


In [4]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# 分类类别
categories = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic','Normal', 'Reconnaissance', 'Shellcode', 'Worms']

# 混淆矩阵（最后一折的结果）
last_cm = confusion_matrix(last_fold_y_true, last_fold_y_pred, labels=range(10))

print("\nLast Fold Confusion Matrix:")
print(last_cm)

report = classification_report(last_fold_y_true, last_fold_y_pred, target_names=categories)
print("\nClassification Report:")
print(report)

# 从混淆矩阵提取所有类别的统计信息
total_samples = last_cm.sum()  # 总样本数
correct_predictions = np.trace(last_cm)  # 对角线元素的和，即正确分类的总数

# 总体准确率（直接计算）
overall_accuracy = correct_predictions / total_samples

# 初始化总体指标
overall_TP = 0
overall_FN = 0
overall_FP = 0
overall_TN = 0


# 重新计算分类报告中的 TP、FP、FN、TN
detection_rates = {}
false_positive_rates = {}

for i, category in enumerate(categories):
    TP = last_cm[i, i]
    FN = last_cm[i, :].sum() - TP
    FP = last_cm[:, i].sum() - TP
    TN = total_samples - (TP + FP + FN)

    if category != "Normal":  # 统计攻击类型的总 TP 和 FN
        overall_TP += TP
        overall_FN += FN
    else:  # 统计正常类型的 TN 和 FP
        overall_TN += TN
        overall_FP += FP

    # 每类检测率和误报率
    detection_rates[category] = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    false_positive_rates[category] = FP / (FP + TN) if (FP + TN) > 0 else 0.0

    print(f"Category: {category}")
    print(f"  Detection Rate (DR): {detection_rates[category]:.4f}")
    print(f"  False Positive Rate (FPR): {false_positive_rates[category]:.4f}")

# 总体检测率（攻击类型的 DR）和误报率（Normal 的 FPR）
overall_dr = overall_TP / (overall_TP + overall_FN) if (overall_TP + overall_FN) > 0 else 0.0
overall_fpr = overall_FP / (overall_FP + overall_TN) if (overall_FP + overall_TN) > 0 else 0.0

print("\nOverall Metrics:")
print(f"  Accuracy (Acc): {overall_accuracy:.4f}")
print(f"  Detection Rate (DR): {overall_dr:.4f}")
print(f"  False Positive Rate (FPR): {overall_fpr:.4f}")


Last Fold Confusion Matrix:
[[  40    0    7  203    4    0   14    0    0    0]
 [   0   26    2  198    3    0    1    3    0    0]
 [   2    7  209 1364   11    6   10   13   11    2]
 [   8   13  135 3942   51   47   77  150   21    9]
 [   0    0    9  271 1566    7  546   13   11    1]
 [   0    0   13   34    4 5828    4    2    2    0]
 [  36    3    6   38  592    4 8592   10   19    0]
 [   0    1   12  253    1    2    6 1122    1    0]
 [   0    2    3    8    6    4    9    4  115    0]
 [   0    0    0    0    0    0    0    0    0   18]]

Classification Report:
                precision    recall  f1-score   support

      Analysis       0.47      0.15      0.23       268
      Backdoor       0.50      0.11      0.18       233
           DoS       0.53      0.13      0.21      1635
      Exploits       0.62      0.89      0.73      4453
       Fuzzers       0.70      0.65      0.67      2424
       Generic       0.99      0.99      0.99      5887
        Normal       0.